In [ ]:
# 1 — Imports

import os
import time
import calendar
import requests
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

In [ ]:
# 2 — Configuración y conexión

load_dotenv()

DB_SERVER   = os.getenv('DB_SERVER')
DB_NAME     = os.getenv('DB_NAME')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

variables = {
    'DB_SERVER'  : DB_SERVER,
    'DB_NAME'    : DB_NAME,
    'DB_USER'    : DB_USER,
    'DB_PASSWORD': DB_PASSWORD
}

faltantes = [k for k, v in variables.items() if not v]
if faltantes:
    raise ValueError(f'Variables de entorno faltantes: {faltantes}')

print('Variables de entorno cargadas correctamente')

start  = time.time()
engine = create_engine(
    f'mssql+pyodbc://{DB_USER}:{DB_PASSWORD}@{DB_SERVER}/{DB_NAME}'
    '?driver=ODBC+Driver+18+for+SQL+Server&TrustServerCertificate=yes'
)

with engine.connect() as conn:
    elapsed = round(time.time() - start, 2)
    print(f'Conexión exitosa | Tiempo: {elapsed}s')

In [ ]:
# 3 — Funciones auxiliares

def verificar_api(url, params):
    """Verifica que el endpoint responde correctamente antes de procesar."""
    response = requests.get(url, headers = {'Accept': 'application/json'}, params = params)

    if response.status_code != 200:
        raise ConnectionError(f'API respondió con status {response.status_code}')
    if not response.text.strip():
        raise ValueError('API respondió con cuerpo vacío')

    return response.json()


def cargar_tabla(df, tabla, anio, mes = None):
    """Elimina el rango e inserta el DataFrame en la tabla indicada."""
    start = time.time()

    schema   = tabla.split('.')[0]
    base     = tabla.split('.')[1]
    all_cols = list(df.columns)

    if mes:
        ultimo_dia = calendar.monthrange(anio, mes)[1]
        rango_ini  = f'{anio}-{mes:02d}-01'
        rango_fin  = f'{anio}-{mes:02d}-{ultimo_dia}'
    else:
        rango_ini  = f'{anio}-01-01'
        rango_fin  = f'{anio}-12-31'

    col_list     = ', '.join([f'[{c}]' for c in all_cols])
    placeholders = ', '.join(['?' for _ in all_cols])
    insert_sql   = f'INSERT INTO [{schema}].[{base}] ({col_list}) VALUES ({placeholders})'
    rows         = [tuple(r) for r in df.itertuples(index=False, name=None)]

    with engine.begin() as conn:
        conn.execute(text(f"""
            DELETE FROM [{schema}].[{base}]
            WHERE Fecha BETWEEN :ini AND :fin
        """), {'ini': rango_ini, 'fin': rango_fin})

        raw = conn.connection.cursor()
        raw.executemany(insert_sql, rows)

    elapsed = round(time.time() - start, 2)
    return elapsed

In [ ]:
# 4 — Generacion

print('=' * 60)
print('GENERACION')
print('=' * 60)

start_total = time.time()
total_filas = 0
anios       = range(2014, datetime.now().year + 1)
url         = 'https://apidatos.ree.es/es/datos/generacion/estructura-generacion'

for anio in anios:
    params = {
        'start_date': f'{anio}-01-01T00:00',
        'end_date'  : f'{anio}-12-31T23:59',
        'time_trunc': 'day',
        'geo_limit' : 'peninsular',
        'geo_ids'   : '8741'
    }

    try:
        datos = verificar_api(url, params).get('included', [])
    except Exception as e:
        print(f'{anio}: error — {e}')
        continue

    registros = []
    for fuente in datos:
        tipo = fuente['attributes']['title']
        for punto in fuente['attributes']['values']:
            registros.append({
                'Fecha'     : punto['datetime'][:10],
                'Fuente'    : tipo,
                'Valor_mwh' : punto['value'],
                'Porcentaje': punto['percentage']
            })

    if not registros:
        print(f'{anio}: sin datos')
        continue

    df          = pd.DataFrame(registros)
    df['Fecha'] = pd.to_datetime(df['Fecha']).dt.date

    elapsed      = cargar_tabla(df, 'ree.Generacion', anio)
    total_filas += len(df)
    print(f'{anio}: {len(df):,} filas | {elapsed}s')

print(f'\nTotal Generacion: {total_filas:,} filas | Tiempo total: {round(time.time() - start_total, 2)}s')

In [ ]:
# 5 — Demanda

print('=' * 60)
print('DEMANDA')
print('=' * 60)

start_total = time.time()
total_filas = 0
url         = 'https://apidatos.ree.es/es/datos/demanda/evolucion'

for anio in anios:
    params = {
        'start_date': f'{anio}-01-01T00:00',
        'end_date'  : f'{anio}-12-31T23:59',
        'time_trunc': 'day',
        'geo_limit' : 'peninsular',
        'geo_ids'   : '8741'
    }

    try:
        datos = verificar_api(url, params).get('included', [])
    except Exception as e:
        print(f'{anio}: error — {e}')
        continue

    registros = []
    for item in datos:
        tipo = item['attributes']['title']
        for punto in item['attributes']['values']:
            registros.append({
                'Fecha'    : punto['datetime'][:10],
                'Tipo'     : tipo,
                'Valor_mwh': punto['value']
            })

    if not registros:
        print(f'{anio}: sin datos')
        continue

    df          = pd.DataFrame(registros)
    df['Fecha'] = pd.to_datetime(df['Fecha']).dt.date

    elapsed      = cargar_tabla(df, 'ree.Demanda', anio)
    total_filas += len(df)
    print(f'{anio}: {len(df):,} filas | {elapsed}s')

print(f'\nTotal Demanda: {total_filas:,} filas | Tiempo total: {round(time.time() - start_total, 2)}s')

In [ ]:
# 6 — Emisiones

print('=' * 60)
print('EMISIONES')
print('=' * 60)

start_total = time.time()
total_filas = 0
url         = 'https://apidatos.ree.es/es/datos/generacion/estructura-generacion-emisiones-asociadas'

for anio in anios:
    params = {
        'start_date': f'{anio}-01-01T00:00',
        'end_date'  : f'{anio}-12-31T23:59',
        'time_trunc': 'day',
        'geo_limit' : 'peninsular',
        'geo_ids'   : '8741'
    }

    try:
        datos = verificar_api(url, params).get('included', [])
    except Exception as e:
        print(f'{anio}: error — {e}')
        continue

    registros = []
    for item in datos:
        tipo = item['attributes']['title']
        for punto in item['attributes']['values']:
            registros.append({
                'Fecha': punto['datetime'][:10],
                'Tipo' : tipo,
                'Valor': punto['value']
            })

    if not registros:
        print(f'{anio}: sin datos')
        continue

    df          = pd.DataFrame(registros)
    df['Fecha'] = pd.to_datetime(df['Fecha']).dt.date

    elapsed      = cargar_tabla(df, 'ree.Emisiones', anio)
    total_filas += len(df)
    print(f'{anio}: {len(df):,} filas | {elapsed}s')

print(f'\nTotal Emisiones: {total_filas:,} filas | Tiempo total: {round(time.time() - start_total, 2)}s')

In [ ]:
# 7 — Precios

print('=' * 60)
print('PRECIOS')
print('=' * 60)

start_total = time.time()
total_filas = 0
url         = 'https://apidatos.ree.es/es/datos/mercados/precios-mercados-tiempo-real'

for anio in range(2014, datetime.now().year + 1):
    for mes in range(1, 13):

        if datetime(anio, mes, 1) > datetime.now():
            continue

        ultimo_dia = calendar.monthrange(anio, mes)[1]
        params     = {
            'start_date': f'{anio}-{mes:02d}-01T00:00',
            'end_date'  : f'{anio}-{mes:02d}-{ultimo_dia}T23:59',
            'time_trunc': 'hour'
        }

        try:
            datos = verificar_api(url, params).get('included', [])
        except Exception as e:
            print(f'{anio}-{mes:02d}: error — {e}')
            continue

        registros = []
        for item in datos:
            tipo = item['attributes']['title']
            for punto in item['attributes']['values']:
                dt   = punto['datetime']
                zona = dt[23:29]
                registros.append({
                    'Fecha'        : dt[:10],
                    'Hora'         : dt[11:19],
                    'Zona'         : zona,
                    'Tipo'         : tipo,
                    'Valor_eur_mwh': punto['value']
                })

        if not registros:
            print(f'{anio}-{mes:02d}: sin datos')
            continue

        df          = pd.DataFrame(registros)
        df['Fecha'] = pd.to_datetime(df['Fecha']).dt.date
        df['Hora']  = pd.to_datetime(df['Hora'], format = '%H:%M:%S').dt.time

        elapsed      = cargar_tabla(df, 'ree.Precios', anio, mes)
        total_filas += len(df)

    print(f'{anio}: acumulado {total_filas:,} filas')

print(f'\nTotal Precios: {total_filas:,} filas | Tiempo total: {round(time.time() - start_total, 2)}s')

In [ ]:
# 8 — Intercambios

print('=' * 60)
print('INTERCAMBIOS')
print('=' * 60)

start_total = time.time()
total_filas = 0
url         = 'https://apidatos.ree.es/es/datos/intercambios/todas-fronteras-programados'

for anio in anios:
    params = {
        'start_date': f'{anio}-01-01T00:00',
        'end_date'  : f'{anio}-12-31T23:59',
        'time_trunc': 'day',
        'geo_limit' : 'peninsular',
        'geo_ids'   : '8741'
    }

    try:
        datos = verificar_api(url, params).get('included', [])
    except Exception as e:
        print(f'{anio}: error — {e}')
        continue

    registros = []
    for pais_item in datos:
        pais    = pais_item['attributes']['title'].capitalize()
        content = pais_item['attributes'].get('content', [])
        for flujo in content:
            tipo = flujo['attributes']['title']
            for punto in flujo['attributes']['values']:
                registros.append({
                    'Fecha'    : punto['datetime'][:10],
                    'Pais'     : pais,
                    'Tipo'     : tipo,
                    'Valor_mwh': punto['value']
                })

    if not registros:
        print(f'{anio}: sin datos')
        continue

    df          = pd.DataFrame(registros)
    df['Fecha'] = pd.to_datetime(df['Fecha']).dt.date

    elapsed      = cargar_tabla(df, 'ree.Intercambios', anio)
    total_filas += len(df)
    print(f'{anio}: {len(df):,} filas | {elapsed}s')

print(f'\nTotal Intercambios: {total_filas:,} filas | Tiempo total: {round(time.time() - start_total, 2)}s')

In [ ]:
# 9 — Resumen final

print('=' * 60)
print('RESUMEN FINAL')
print('=' * 60)

start = time.time()

with engine.connect() as conn:
    resultado  = conn.execute(text("""
        SELECT 'Generacion'    AS Tabla, COUNT(*) AS Filas, MIN(Fecha) AS Desde, MAX(Fecha) AS Hasta FROM ree.Generacion
        UNION ALL SELECT 'Demanda',      COUNT(*), MIN(Fecha), MAX(Fecha) FROM ree.Demanda
        UNION ALL SELECT 'Emisiones',    COUNT(*), MIN(Fecha), MAX(Fecha) FROM ree.Emisiones
        UNION ALL SELECT 'Precios',      COUNT(*), MIN(Fecha), MAX(Fecha) FROM ree.Precios
        UNION ALL SELECT 'Intercambios', COUNT(*), MIN(Fecha), MAX(Fecha) FROM ree.Intercambios
    """))
    df_resumen = pd.DataFrame(resultado.fetchall(), columns = ['Tabla', 'Filas', 'Desde', 'Hasta'])

print(df_resumen.to_string(index = False))
print(f'\nTiempo consulta: {round(time.time() - start, 2)}s')